# Rekayasa Fitur — DataCo SMART SUPPLY CHAIN

Mengubah dataset mentah DataCo menjadi dataframe yang kaya fitur untuk pemodelan prediktif **simulasi rantai pasok pendingin**, **risiko logistik**, dan **degradasi kualitas**.

| Langkah | Fitur | Rumus |
|------|---------|----------|
| 1 | Pra-pemrosesan | Hapus ID, data pribadi, tautan gambar, target leakage |
| 2 | Keterlambatan | `Hari aktual − Hari terjadwal` |
| 3 | Jarak | Haversine ke gudang (39.8283, −98.5795) |
| 4 | Normalisasi | `MinMaxScaler` sklearn |
| 5 | Risiko Rute | `0.4·Keterlambatan_norm + 0.3·Jarak_norm + 0.3·Late_delivery_risk` |
| 6 | Deviasi Suhu | `0.002·Jarak_norm + 0.5·Keterlambatan_norm + 0.3·Risiko Rute` |
| 7 | Degradasi Kualitas | `Q0 · exp(-λ · (Keterlambatan_norm + Deviasi Suhu))` |
| 8 | Biaya Pendinginan | `Jarak × FaktorModePengiriman` |

---
## Impor Modul

In [15]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

---
## Konfigurasi & Konstanta

In [16]:
# --- Jalur Direktori ---
DATA_PATH   = "./data/DataCoSupplyChainDataset.csv"
OUTPUT_PATH = "./data/processed/cold_chain_data.csv"

# --- Gudang Referensi (pusat geografis AS) ---
WAREHOUSE_LAT = 39.8283
WAREHOUSE_LON = -98.5795

# --- Jari-jari rata-rata Bumi (WGS-84) ---
EARTH_RADIUS_KM = 6_371.0

# --- Model degradasi kualitas ---
Q0     = 100    # Kualitas awal produk (skala 0-100)
LAMBDA = 0.05   # Konstanta peluruhan

# --- Pengali biaya pendinginan per mode pengiriman ---
# Mode pengiriman lebih cepat membutuhkan pendinginan aktif -> faktor biaya lebih tinggi
REFRIGERATION_FACTORS = {
    "Same Day"       : 3.0,
    "First Class"    : 2.5,
    "Second Class"   : 2.0,
    "Standard Class" : 1.0,
}

# --- Kolom yang dihapus (ID, data pribadi, gambar, target leakage) ---
COLUMNS_TO_DROP = [
    "Customer Email", "Customer Fname", "Customer Lname",
    "Customer Password", "Customer Street", "Customer Zipcode",
    "Customer Id", "Order Customer Id", "Order Id",
    "Order Item Id", "Order Item Cardprod Id",
    "Product Card Id", "Product Image", "Product Description",
    "Product Status",       # jarang / tidak informatif
    "Category Id",          # redundan dengan Nama Kategori
    "Department Id",        # redundan dengan Nama Departemen
    "Product Category Id",  # redundan dengan Nama Kategori
    "Order Zipcode",        # redundan dengan Kota/Provinsi Pesanan
    "Delivery Status",      # target leakage (diturunkan dari Late_delivery_risk)
]

print(f"Konfigurasi dimuat. Akan menghapus {len(COLUMNS_TO_DROP)} kolom yang tidak relevan.")

Konfigurasi dimuat. Akan menghapus 20 kolom yang tidak relevan.


---
## Memuat & Pra-pemrosesan Data

In [17]:
# Memuat dataset mentah
df = pd.read_csv(DATA_PATH, encoding_errors="ignore")
print(f"Bentuk awal: {df.shape}")
print(f"Kolom : {list(df.columns)}")

Bentuk awal: (180519, 53)
Kolom : ['Type', 'Days for shipping (real)', 'Days for shipment (scheduled)', 'Benefit per order', 'Sales per customer', 'Delivery Status', 'Late_delivery_risk', 'Category Id', 'Category Name', 'Customer City', 'Customer Country', 'Customer Email', 'Customer Fname', 'Customer Id', 'Customer Lname', 'Customer Password', 'Customer Segment', 'Customer State', 'Customer Street', 'Customer Zipcode', 'Department Id', 'Department Name', 'Latitude', 'Longitude', 'Market', 'Order City', 'Order Country', 'Order Customer Id', 'order date (DateOrders)', 'Order Id', 'Order Item Cardprod Id', 'Order Item Discount', 'Order Item Discount Rate', 'Order Item Id', 'Order Item Product Price', 'Order Item Profit Ratio', 'Order Item Quantity', 'Sales', 'Order Item Total', 'Order Profit Per Order', 'Order Region', 'Order State', 'Order Status', 'Order Zipcode', 'Product Card Id', 'Product Category Id', 'Product Description', 'Product Image', 'Product Name', 'Product Price', 'Product

In [18]:
# Menghapus kolom yang tidak relevan (hanya yang ada di dataframe)
cols_present = [c for c in COLUMNS_TO_DROP if c in df.columns]
df.drop(columns=cols_present, inplace=True)

print(f"{len(cols_present)} kolom dihapus")
print(f"Bentuk tersisa: {df.shape}")
df.head(3)

20 kolom dihapus
Bentuk tersisa: (180519, 33)


,Type,Days for shipping (real),Days for shipment (scheduled),Benefit per order,Sales per customer,Late_delivery_risk,Category Name,Customer City,Customer Country,Customer Segment,...,Sales,Order Item Total,Order Profit Per Order,Order Region,Order State,Order Status,Product Name,Product Price,shipping date (DateOrders),Shipping Mode
0,DEBIT,3,4,91.250000,314.640015,0,Sporting Goods,Caguas,Puerto Rico,Consumer,...,327.75,314.640015,91.250000,Southeast Asia,Java Occidental,COMPLETE,Smart watch,327.75,2/3/2018 22:56,Standard Class
1,TRANSFER,5,4,-249.089996,311.359985,1,Sporting Goods,Caguas,Puerto Rico,Consumer,...,327.75,311.359985,-249.089996,South Asia,Rajastn,PENDING,Smart watch,327.75,1/18/2018 12:27,Standard Class
2,CASH,4,4,-247.779999,309.720001,0,Sporting Goods,San Jose,EE. UU.,Consumer,...,327.75,309.720001,-247.779999,South Asia,Rajastn,CLOSED,Smart watch,327.75,1/17/2018 12:06,Standard Class


---
## Perhitungan Keterlambatan

$$\text{Delay} = \text{Days for shipping (real)} - \text{Days for shipment (scheduled)}$$

Keterlambatan positif berarti pengiriman tiba **lebih lambat** dari yang dijadwalkan (pelanggaran SLA).

In [19]:
df["Delay"] = (
    df["Days for shipping (real)"] - df["Days for shipment (scheduled)"]
)

print(f"Rentang Keterlambatan: [{df['Delay'].min()}, {df['Delay'].max()}]")
print(f"Rata-rata Keterlambatan : {df['Delay'].mean():.4f} hari")
df["Delay"].value_counts().sort_index()

Rentang Keterlambatan: [-2, 4]
Rata-rata Keterlambatan : 0.5658 hari


Delay
-2    21666
-1    21700
 0    33753
 1    60647
 2    28718
 3     7052
 4     6983
Name: count, dtype: int64

---
## Perhitungan Jarak (Rumus Haversine)

Jarak lingkaran besar antara setiap lokasi pelanggan dan gudang referensi di **(39.8283, -98.5795)**.

$$a = \sin^2\!\left(\frac{\Delta\varphi}{2}\right) + \cos(\varphi_1)\cos(\varphi_2)\sin^2\!\left(\frac{\Delta\lambda}{2}\right)$$
$$d = 2R \cdot \arcsin\!\left(\sqrt{a}\right)$$

In [20]:
def haversine(lat1, lon1, lat2, lon2):
    """
    Menghitung jarak lingkaran besar menggunakan rumus Haversine.
    
    Parameters
    ----------
    lat1, lon1 : array-like  — Titik 1 dalam DERAJAT
    lat2, lon2 : array-like  — Titik 2 dalam DERAJAT
    
    Returns
    -------
    np.ndarray — Jarak dalam kilometer
    """
    # Derajat -> Radian
    phi1, lam1, phi2, lam2 = map(np.radians, [lat1, lon1, lat2, lon2])
    
    d_phi = phi2 - phi1
    d_lam = lam2 - lam1
    
    a = (
        np.sin(d_phi / 2.0) ** 2
        + np.cos(phi1) * np.cos(phi2) * np.sin(d_lam / 2.0) ** 2
    )
    c = 2.0 * np.arcsin(np.sqrt(a))
    
    return EARTH_RADIUS_KM * c

print("Fungsi Haversine telah didefinisikan.")

Fungsi Haversine telah didefinisikan.


In [21]:
df["Distance"] = haversine(
    df["Latitude"].values,
    df["Longitude"].values,
    WAREHOUSE_LAT,
    WAREHOUSE_LON,
)

print(f"Rentang Jarak: [{df['Distance'].min():.2f}, {df['Distance'].max():.2f}] km")
print(f"Rata-rata Jarak : {df['Distance'].mean():.2f} km")
df["Distance"].describe()

Rentang Jarak: [51.00, 14501.57] km
Rata-rata Jarak : 2525.20 km


count    180519.000000
mean       2525.202039
std        1234.153011
min          50.996441
25%        1662.096903
50%        2059.502214
75%        3907.526298
max       14501.565200
Name: Distance, dtype: float64

---
## Normalisasi (MinMaxScaler)

Skalakan **Keterlambatan** (dibatasi di 0, sehingga kedatangan awal tidak memengaruhi rentang) dan **Jarak** ke rentang [0, 1].

In [22]:
scaler = MinMaxScaler()

# Batasi Keterlambatan >= 0 sebelum scaling
delay_clipped = df["Delay"].clip(lower=0).values.reshape(-1, 1)
distance_vals = df["Distance"].values.reshape(-1, 1)

# Fit-transform kedua fitur secara bersamaan
scaled = scaler.fit_transform(
    np.hstack([delay_clipped, distance_vals])
)

df["Delay_norm"]    = scaled[:, 0]
df["Distance_norm"] = scaled[:, 1]

print("Normalisasi selesai.")
df[["Delay", "Delay_norm", "Distance", "Distance_norm"]].describe().round(4)

Normalisasi selesai.


,Delay,Delay_norm,Distance,Distance_norm
count,180519.0000,180519.0000,180519.0000,180519.0000
mean,0.5658,0.2315,2525.2020,0.1712
std,1.4910,0.2604,1234.1530,0.0854
min,-2.0000,0.0000,50.9964,0.0000
25%,0.0000,0.0000,1662.0969,0.1115
50%,1.0000,0.2500,2059.5022,0.1390
75%,1.0000,0.2500,3907.5263,0.2669
max,4.0000,1.0000,14501.5652,1.0000


---
## Indeks Risiko Rute

Skor risiko komposit yang menggabungkan tingkat keparahan keterlambatan, paparan geografis, dan indikator keterlambatan pengiriman:

$$\text{RouteRisk} = 0.4 \times \text{Delay}_{\text{norm}} + 0.3 \times \text{Distance}_{\text{norm}} + 0.3 \times \text{Late\_delivery\_risk}$$

In [23]:
df["RouteRisk"] = (
    0.4 * df["Delay_norm"]
    + 0.3 * df["Distance_norm"]
    + 0.3 * df["Late_delivery_risk"]
)

print(f"Rentang Risiko Rute: [{df['RouteRisk'].min():.4f}, {df['RouteRisk'].max():.4f}]")
df["RouteRisk"].describe()

Rentang Risiko Rute: [0.0000, 0.9445]


count    180519.000000
mean          0.308459
std           0.237801
min           0.000000
25%           0.046266
50%           0.424686
75%           0.480151
max           0.944512
Name: RouteRisk, dtype: float64

---
## Simulasi Rantai Pasok Pendingin

### Deviasi Suhu
Proksi seberapa jauh suhu kargo menyimpang dari titik ideal:

$$\text{TempDev} = 0.002 \times \text{Distance}_{\text{norm}} + 0.5 \times \text{Delay}_{\text{norm}} + 0.3 \times \text{RouteRisk}$$

### Degradasi Kualitas (Peluruhan Eksponensial)
Model degradasi kinetik orde pertama (Labuza & Riboh, 1982):

$$Q(t) = Q_0 \cdot e^{-\lambda \cdot (\text{Delay}_{\text{norm}} + \text{TempDev})}$$

di mana $Q_0 = 100$ dan $\lambda = 0.05$.

In [24]:
# Deviasi Suhu
df["TempDev"] = (
    0.002 * df["Distance_norm"]
    + 0.5  * df["Delay_norm"]
    + 0.3  * df["RouteRisk"]
)

print(f"Rentang Deviasi Suhu: [{df['TempDev'].min():.4f}, {df['TempDev'].max():.4f}]")
df["TempDev"].describe()

Rentang Deviasi Suhu: [0.0000, 0.7850]


count    180519.000000
mean          0.208637
std           0.196785
min           0.000000
25%           0.014188
50%           0.253211
75%           0.269703
max           0.784984
Name: TempDev, dtype: float64

In [25]:
# Degradasi Kualitas (peluruhan eksponensial)
df["QualityDegradation"] = (
    Q0 * np.exp(-LAMBDA * (df["Delay_norm"] + df["TempDev"]))
).clip(lower=0, upper=100)

print(f"Rentang Kualitas: [{df['QualityDegradation'].min():.4f}, {df['QualityDegradation'].max():.4f}]")
df["QualityDegradation"].describe()

Rentang Kualitas: [91.4618, 100.0000]


count    180519.000000
mean         97.848499
std           2.205418
min          91.461763
25%          97.434957
50%          97.515336
75%          99.929084
max         100.000000
Name: QualityDegradation, dtype: float64

---
## Biaya Pendinginan

$$\text{RefrigerationCost} = \text{Distance} \times \text{Factor}_{\text{mode}}$$

| Mode Pengiriman | Faktor | Rasional |
|---------------|--------|-----------|
| Same Day | 3.0 | Toleransi rantai pasok pendingin paling ketat |
| First Class | 2.5 | Pendinginan aktif prioritas tinggi |
| Second Class | 2.0 | Pendinginan aktif standar |
| Standard Class | 1.0 | Pendinginan pasif / minimal |

In [26]:
df["RefrigerationCost"] = (
    df["Distance"]
    * df["Shipping Mode"].map(REFRIGERATION_FACTORS)
)

print("Biaya pendinginan berdasarkan mode pengiriman:")
df.groupby("Shipping Mode")["RefrigerationCost"].describe().round(2)

Biaya pendinginan berdasarkan mode pengiriman:

,count,mean,std,min,25%,50%,75%,max
Shipping Mode,,,,,,,,
First Class,27814.0,6349.87,3079.09,127.49,4267.55,5162.48,9768.94,31178.30
Same Day,9737.0,7425.85,3582.90,152.99,4852.88,6128.57,11721.91,20460.32
Second Class,35216.0,5021.69,2472.60,101.99,3135.84,4094.70,7814.84,23657.62
Standard Class,107752.0,2530.60,1237.50,51.00,1662.10,2060.76,3907.55,14501.57


---
## Dataset Akhir — Ekspor & Ringkasan

In [27]:
# Pemilihan kolom akhir
FINAL_OUTPUT_COLUMNS = [
    # --- Kolom asli ---
    "order date (DateOrders)",
    "Order Item Quantity",
    "Sales",
    "Shipping Mode",
    "Market",
    "Category Name",
    "Order Region",
    "Product Price",
    "Latitude",
    "Longitude",
    "Days for shipping (real)",
    "Days for shipment (scheduled)",
    "Late_delivery_risk",
    # --- Fitur hasil rekayasa ---
    "Delay",
    "Distance",
    "Delay_norm",
    "Distance_norm",
    "RouteRisk",
    "TempDev",
    "QualityDegradation",
    "RefrigerationCost",
]

# Memverifikasi semua kolom ada
missing = [c for c in FINAL_OUTPUT_COLUMNS if c not in df.columns]
if missing:
    raise KeyError(f"Kolom tidak ditemukan: {missing}")

df_final = df[FINAL_OUTPUT_COLUMNS].copy()
print(f"Bentuk akhir: {df_final.shape}")
df_final.head()

Bentuk akhir: (180519, 21)


,order date (DateOrders),Order Item Quantity,Sales,Shipping Mode,Market,Category Name,Order Region,Product Price,Latitude,Longitude,...,Days for shipment (scheduled),Late_delivery_risk,Delay,Distance,Delay_norm,Distance_norm,RouteRisk,TempDev,QualityDegradation,RefrigerationCost
0,1/31/2018 22:56,1,327.75,Standard Class,Pacific Asia,Sporting Goods,Southeast Asia,327.75,18.251453,-66.037056,...,4,0,-1,3933.139271,0.00,0.268650,0.080595,0.024716,99.876497,3933.139271
1,1/13/2018 12:27,1,327.75,Standard Class,Pacific Asia,Sporting Goods,South Asia,327.75,18.279451,-66.037064,...,4,1,1,3930.958376,0.25,0.268499,0.480550,0.269702,97.434961,3930.958376
2,1/13/2018 12:06,1,327.75,Standard Class,Pacific Asia,Sporting Goods,South Asia,327.75,37.292233,-121.881279,...,4,0,0,2039.700242,0.00,0.137621,0.041286,0.012661,99.936714,2039.700242
3,1/13/2018 11:45,1,327.75,Standard Class,Pacific Asia,Sporting Goods,Oceania,327.75,34.125946,-118.291016,...,4,0,-1,1857.463232,0.00,0.125010,0.037503,0.011501,99.942512,1857.463232
4,1/13/2018 11:24,1,327.75,Standard Class,Pacific Asia,Sporting Goods,Oceania,327.75,18.253769,-66.037048,...,4,0,-2,3932.959485,0.00,0.268637,0.080591,0.024715,99.876503,3932.959485


In [28]:
# Simpan ke CSV
df_final.to_csv(OUTPUT_PATH, index=False)
print(f"Dataset disimpan di: {OUTPUT_PATH}")
print(f"Bentuk: {df_final.shape}")

Dataset disimpan di: ./data/processed/cold_chain_data.csv
Bentuk: (180519, 21)


In [29]:
# Statistik fitur hasil rekayasa
derived = [
    "Delay", "Distance", "Delay_norm", "Distance_norm",
    "RouteRisk", "TempDev", "QualityDegradation", "RefrigerationCost",
]

print("=== Statistik Fitur Hasil Rekayasa ===")
display(df_final[derived].describe().round(4))

=== Statistik Fitur Hasil Rekayasa ===


,Delay,Distance,Delay_norm,Distance_norm,RouteRisk,TempDev,QualityDegradation,RefrigerationCost
count,180519.0000,180519.0000,180519.0000,180519.0000,180519.0000,180519.0000,180519.0000,180519.0000
mean,0.5658,2525.2020,0.2315,0.1712,0.3085,0.2086,97.8485,3869.0758
std,1.4910,1234.1530,0.2604,0.0854,0.2378,0.1968,2.2054,2686.8203
min,-2.0000,50.9964,0.0000,0.0000,0.0000,0.0000,91.4618,50.9964
25%,0.0000,1662.0969,0.0000,0.1115,0.0463,0.0142,97.4350,1865.8780
50%,1.0000,2059.5022,0.2500,0.1390,0.4247,0.2532,97.5153,3904.4439
75%,1.0000,3907.5263,0.2500,0.2669,0.4802,0.2697,99.9291,4179.6246
max,4.0000,14501.5652,1.0000,1.0000,0.9445,0.7850,100.0000,31178.2960


In [30]:
# Rata-rata fitur hasil rekayasa berdasarkan Pasar
print("=== Rata-rata berdasarkan Pasar ===")
display(df_final.groupby("Market")[derived].mean().round(4))

=== Rata-rata berdasarkan Pasar ===


,Delay,Distance,Delay_norm,Distance_norm,RouteRisk,TempDev,QualityDegradation,RefrigerationCost
Market,,,,,,,,
Africa,0.5600,2470.1164,0.2275,0.1674,0.3050,0.2056,97.8827,3745.1305
Europe,0.5708,2548.7210,0.2331,0.1728,0.3107,0.2101,97.8334,3912.0695
LATAM,0.5578,2527.6275,0.2304,0.1714,0.3066,0.2075,97.8595,3853.5254
Pacific Asia,0.5694,2520.4242,0.2324,0.1709,0.3094,0.2094,97.8407,3871.9831
USCA,0.5689,2506.9798,0.2311,0.1700,0.3078,0.2082,97.8529,3867.5771


In [31]:
df_final.head(2)

,order date (DateOrders),Order Item Quantity,Sales,Shipping Mode,Market,Category Name,Order Region,Product Price,Latitude,Longitude,...,Days for shipment (scheduled),Late_delivery_risk,Delay,Distance,Delay_norm,Distance_norm,RouteRisk,TempDev,QualityDegradation,RefrigerationCost
0,1/31/2018 22:56,1,327.75,Standard Class,Pacific Asia,Sporting Goods,Southeast Asia,327.75,18.251453,-66.037056,...,4,0,-1,3933.139271,0.00,0.268650,0.080595,0.024716,99.876497,3933.139271
1,1/13/2018 12:27,1,327.75,Standard Class,Pacific Asia,Sporting Goods,South Asia,327.75,18.279451,-66.037064,...,4,1,1,3930.958376,0.25,0.268499,0.480550,0.269702,97.434961,3930.958376


In [32]:
df_final.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 180519 entries, 0 to 180518
Data columns (total 21 columns):
 #   Column                         Non-Null Count   Dtype  
---  ------                         --------------   -----  
 0   order date (DateOrders)        180519 non-null  object 
 1   Order Item Quantity            180519 non-null  int64  
 2   Sales                          180519 non-null  float64
 3   Shipping Mode                  180519 non-null  object 
 4   Market                         180519 non-null  object 
 5   Category Name                  180519 non-null  object 
 6   Order Region                   180519 non-null  object 
 7   Product Price                  180519 non-null  float64
 8   Latitude                       180519 non-null  float64
 9   Longitude                      180519 non-null  float64
 10  Days for shipping (real)       180519 non-null  int64  
 11  Days for shipment (scheduled)  180519 non-null  int64  
 12  Late_delivery_risk            

---
## Signifikansi Fitur untuk Optimasi Rantai Pasok

| Fitur | Mengapa Ini Penting |
|---------|----------------|
| **Keterlambatan** | Secara langsung mengukur kinerja pengiriman. Nilai positif menandai pelanggaran SLA, memungkinkan evaluasi perusahaan ekspedisi. |
| **Jarak** (Haversine) | Proksi berbasis fisika untuk waktu transit, konsumsi bahan bakar, dan paparan rantai pasok pendingin. Rute yang lebih panjang meningkatkan probabilitas fluktuasi suhu. |
| **Risiko Rute** | Menggabungkan tingkat keparahan keterlambatan, paparan geografis, dan risiko keterlambatan pengiriman ke dalam satu skor. Skor berkelanjutan ini lebih informatif untuk model ML berbasis gradien (Hastie et al., 2009). |
| **Deviasi Suhu** | Mensimulasikan ketidakstabilan rantai pasok pendingin. Dalam rantai pasok makanan/farmasi, setiap tambahan derajat-jam penyimpangan mempercepat pembusukan secara non-linear (Mercier et al., 2017). |
| **Degradasi Kualitas** | Memodelkan degradasi kinetik orde pertama $Q_0 \cdot e^{-\lambda t}$ (Labuza & Riboh, 1982). Output pada skala 0-100 berfungsi sebagai target regresi atau ambang klasifikasi. |
| **Biaya Pendinginan** | Menangkap kompromi antara biaya dan kecepatan serta intensitas pendinginan. Pengiriman di hari yang sama membutuhkan pendinginan aktif dengan toleransi yang lebih ketat, meningkatkan biaya per-km. Ini memungkinkan optimasi total biaya kepemilikan. |